# NumPy Neural Network From Scratch

This notebook is based on a 2017 learning exercise I did where we were asked to
derive a small fully connected neural network and implement it directly in
NumPy. The point is to show that a neural network is just matrix algebra plus the chain rule.


## The Model

For each layer `l`, the forward pass is:

```text
Z[l] = W[l] A[l-1] + b[l]
A[l] = g[l](Z[l])
```

The hidden layers use ReLU. The final layer uses a sigmoid so that the output is
a probability for binary classification.

For binary cross-entropy:

```text
J = -(1/m) sum( Y log(AL) + (1 - Y) log(1 - AL) )
```

Backpropagation follows from the chain rule. For a linear layer:

```text
dW[l]      = (1/m) dZ[l] A[l-1].T
db[l]      = (1/m) sum(dZ[l])
dA[l-1]    = W[l].T dZ[l]
```


In [ ]:
import numpy as np

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

np.set_printoptions(precision=4, suppress=True)


## A Small Synthetic Dataset

To keep things simple, I build a small, synthetic dataset. 


In [ ]:
def make_two_moons(n_samples=800, noise=0.16, seed=7):
    """Generate a two-moons binary classification dataset using only NumPy."""
    rng = np.random.default_rng(seed)
    n_top = n_samples // 2
    n_bottom = n_samples - n_top

    theta_top = rng.uniform(0.0, np.pi, n_top)
    theta_bottom = rng.uniform(0.0, np.pi, n_bottom)

    top = np.stack([np.cos(theta_top), np.sin(theta_top)], axis=0)
    bottom = np.stack([1.0 - np.cos(theta_bottom), 0.5 - np.sin(theta_bottom)], axis=0)

    X = np.concatenate([top, bottom], axis=1)
    Y = np.concatenate([np.zeros(n_top), np.ones(n_bottom)]).reshape(1, -1)

    X = X + rng.normal(scale=noise, size=X.shape)
    X = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)

    order = rng.permutation(n_samples)
    return X[:, order], Y[:, order]


X, Y = make_two_moons()
split = 600
X_train, Y_train = X[:, :split], Y[:, :split]
X_test, Y_test = X[:, split:], Y[:, split:]

print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_test shape: ", X_test.shape)
print("Y_test shape: ", Y_test.shape)


In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(X_train[0], X_train[1], c=Y_train.ravel(), s=16, cmap="coolwarm", alpha=0.8)
    ax.set_title("Generated two-moons training data")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_aspect("equal", adjustable="box")
    plt.show()
else:
    print("matplotlib is not installed; skipping the dataset plot.")


## Forward Pass Helpers

The implementation below keeps the functions small on purpose. Each one maps
closely to a mathematical object: activation functions, parameter
initialization, a linear layer, a linear-plus-activation layer, and finally the
whole model.


In [ ]:
def sigmoid(Z):
    return 1.0 / (1.0 + np.exp(-np.clip(Z, -500, 500)))


def relu(Z):
    return np.maximum(0.0, Z)


def initialize_parameters(layer_dims, seed=3):
    """He-initialize W matrices and zero-initialize b vectors."""
    rng = np.random.default_rng(seed)
    parameters = {}

    for l in range(1, len(layer_dims)):
        parameters[f"W{l}"] = rng.normal(
            loc=0.0,
            scale=np.sqrt(2.0 / layer_dims[l - 1]),
            size=(layer_dims[l], layer_dims[l - 1]),
        )
        parameters[f"b{l}"] = np.zeros((layer_dims[l], 1))

    return parameters


def linear_forward(A_prev, W, b):
    Z = W @ A_prev + b
    return Z, (A_prev, W, b)


def activation_forward(A_prev, W, b, activation):
    Z, linear_cache = linear_forward(A_prev, W, b)

    if activation == "relu":
        A = relu(Z)
    elif activation == "sigmoid":
        A = sigmoid(Z)
    else:
        raise ValueError(f"unknown activation: {activation}")

    return A, (linear_cache, Z, activation)


def model_forward(X, parameters):
    A = X
    caches = []
    L = len(parameters) // 2

    for l in range(1, L):
        A, cache = activation_forward(A, parameters[f"W{l}"], parameters[f"b{l}"], "relu")
        caches.append(cache)

    AL, cache = activation_forward(A, parameters[f"W{L}"], parameters[f"b{L}"], "sigmoid")
    caches.append(cache)
    return AL, caches


def compute_cost(AL, Y):
    eps = 1e-12
    AL = np.clip(AL, eps, 1.0 - eps)
    cost = -np.mean(Y * np.log(AL) + (1 - Y) * np.log(1 - AL))
    return float(np.squeeze(cost))


## Backpropagation

The backward pass mirrors the forward pass. Each cached layer remembers enough
state to pass gradients backward through the network.


In [ ]:
def linear_backward(dZ, cache):
    A_prev, W, b = cache
    m = A_prev.shape[1]

    dW = (dZ @ A_prev.T) / m
    db = np.sum(dZ, axis=1, keepdims=True) / m
    dA_prev = W.T @ dZ
    return dA_prev, dW, db


def activation_backward(dA, cache):
    linear_cache, Z, activation = cache

    if activation == "relu":
        dZ = dA * (Z > 0)
    elif activation == "sigmoid":
        A = sigmoid(Z)
        dZ = dA * A * (1 - A)
    else:
        raise ValueError(f"unknown activation: {activation}")

    return linear_backward(dZ, linear_cache)


def model_backward(AL, Y, caches):
    grads = {}
    L = len(caches)
    eps = 1e-12
    AL_safe = np.clip(AL, eps, 1.0 - eps)

    dAL = -(Y / AL_safe - (1 - Y) / (1 - AL_safe))

    dA_prev, dW, db = activation_backward(dAL, caches[-1])
    grads[f"dA{L - 1}"] = dA_prev
    grads[f"dW{L}"] = dW
    grads[f"db{L}"] = db

    for l in reversed(range(L - 1)):
        dA_prev, dW, db = activation_backward(grads[f"dA{l + 1}"], caches[l])
        grads[f"dA{l}"] = dA_prev
        grads[f"dW{l + 1}"] = dW
        grads[f"db{l + 1}"] = db

    return grads


def update_parameters(parameters, grads, learning_rate):
    updated = {key: value.copy() for key, value in parameters.items()}
    L = len(parameters) // 2

    for l in range(1, L + 1):
        updated[f"W{l}"] -= learning_rate * grads[f"dW{l}"]
        updated[f"b{l}"] -= learning_rate * grads[f"db{l}"]

    return updated


## Gradient Check

Before training, it is worth checking that the analytic gradients from
backpropagation agree with numerical gradients from finite differences.

This is slow, so we only run it on a tiny network.


In [ ]:
def pack_parameters(parameters):
    pieces = []
    metadata = []
    L = len(parameters) // 2

    for l in range(1, L + 1):
        for name in (f"W{l}", f"b{l}"):
            array = parameters[name]
            pieces.append(array.reshape(-1, 1))
            metadata.append((name, array.shape, array.size))

    return np.vstack(pieces), metadata


def unpack_parameters(theta, metadata):
    parameters = {}
    start = 0

    for name, shape, size in metadata:
        parameters[name] = theta[start : start + size].reshape(shape)
        start += size

    return parameters


def pack_gradients(grads, L):
    pieces = []
    for l in range(1, L + 1):
        pieces.append(grads[f"dW{l}"].reshape(-1, 1))
        pieces.append(grads[f"db{l}"].reshape(-1, 1))
    return np.vstack(pieces)


def gradient_check(layer_dims=(2, 3, 1), n_examples=5, epsilon=1e-7, seed=11):
    rng = np.random.default_rng(seed)
    X_small = rng.normal(size=(layer_dims[0], n_examples))
    Y_small = (rng.random((1, n_examples)) > 0.5).astype(float)

    parameters = initialize_parameters(layer_dims, seed=seed + 1)
    AL, caches = model_forward(X_small, parameters)
    grads = model_backward(AL, Y_small, caches)

    theta, metadata = pack_parameters(parameters)
    grad_vector = pack_gradients(grads, len(layer_dims) - 1)
    numerical_grad = np.zeros_like(theta)

    for i in range(theta.size):
        theta_plus = theta.copy()
        theta_minus = theta.copy()
        theta_plus[i, 0] += epsilon
        theta_minus[i, 0] -= epsilon

        plus_params = unpack_parameters(theta_plus, metadata)
        minus_params = unpack_parameters(theta_minus, metadata)
        J_plus = compute_cost(model_forward(X_small, plus_params)[0], Y_small)
        J_minus = compute_cost(model_forward(X_small, minus_params)[0], Y_small)
        numerical_grad[i, 0] = (J_plus - J_minus) / (2 * epsilon)

    numerator = np.linalg.norm(grad_vector - numerical_grad)
    denominator = np.linalg.norm(grad_vector) + np.linalg.norm(numerical_grad) + 1e-12
    return numerator / denominator


difference = gradient_check()
print(f"gradient-check relative difference: {difference:.3e}")


## Train The Network

A small `[2, 8, 4, 1]` network is enough for the synthetic two-moons problem.
The implementation uses full-batch gradient descent to keep the mechanics easy
to inspect.


In [ ]:
def predict(X, parameters):
    AL, _ = model_forward(X, parameters)
    return (AL >= 0.5).astype(int), AL


def accuracy(X, Y, parameters):
    labels, _ = predict(X, parameters)
    return float(np.mean(labels == Y))


def train(X, Y, layer_dims, learning_rate=0.25, iterations=4000, seed=4, log_every=500):
    parameters = initialize_parameters(layer_dims, seed=seed)
    history = []

    for step in range(iterations + 1):
        AL, caches = model_forward(X, parameters)
        cost = compute_cost(AL, Y)
        grads = model_backward(AL, Y, caches)
        parameters = update_parameters(parameters, grads, learning_rate)

        if step % log_every == 0 or step == iterations:
            history.append((step, cost))

    return parameters, history


layer_dims = [2, 8, 4, 1]
parameters, history = train(X_train, Y_train, layer_dims)

for step, cost in history:
    print(f"step {step:4d} | cost {cost:.4f}")

print(f"train accuracy: {accuracy(X_train, Y_train, parameters):.3f}")
print(f"test accuracy:  {accuracy(X_test, Y_test, parameters):.3f}")


In [ ]:
if HAS_MATPLOTLIB:
    steps, costs = zip(*history)
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.plot(steps, costs, marker="o")
    ax.set_title("Training cost")
    ax.set_xlabel("Gradient descent step")
    ax.set_ylabel("Binary cross-entropy")
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("matplotlib is not installed; skipping the cost plot.")


In [ ]:
def plot_decision_boundary(X, Y, parameters, grid_steps=240):
    if not HAS_MATPLOTLIB:
        print("matplotlib is not installed; skipping the decision-boundary plot.")
        return

    x_min, x_max = X[0].min() - 0.6, X[0].max() + 0.6
    y_min, y_max = X[1].min() - 0.6, X[1].max() + 0.6
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_steps),
        np.linspace(y_min, y_max, grid_steps),
    )
    grid = np.vstack([xx.ravel(), yy.ravel()])
    _, probs = predict(grid, parameters)
    zz = probs.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.contourf(xx, yy, zz, levels=30, cmap="coolwarm", alpha=0.35)
    ax.contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=1.5)
    ax.scatter(X[0], X[1], c=Y.ravel(), s=16, cmap="coolwarm", edgecolors="white", linewidths=0.3)
    ax.set_title("Learned decision boundary")
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_aspect("equal", adjustable="box")
    plt.show()


plot_decision_boundary(X_train, Y_train, parameters)


## Lessons Learned

- each layer is an affine map plus a nonlinear activation;
- the forward pass caches the values needed by the backward pass;
- backpropagation is the chain rule applied from right to left;
- vectorization turns the same equations into efficient array operations;
- a numerical gradient check is a practical way to catch algebra mistakes.
